# FER2013 Training Experiments

This notebook is designed for Colab or Kaggle GPU training. It compares the model settings requested by the instructor: baseline CNN, augmentation, frozen transfer learning, fine-tuning, and optional imbalance handling.

## 1. Setup

For Colab, clone your GitHub repository or upload it to Drive. For Kaggle, add the FER2013 image-folder dataset and set `DATA_DIR` to the Kaggle input path.


In [ ]:
# Optional in Colab/Kaggle if dependencies are missing:
# !pip install -q torch torchvision pandas scikit-learn matplotlib seaborn tqdm

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

# Local path after extracting Kaggle data:
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'fer2013_images'

# Kaggle example, uncomment and edit if needed:
# DATA_DIR = Path('/kaggle/input/fer2013')

RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)
DATA_DIR


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch

from fer_project.data import TransformConfig, build_imagefolder_dataloaders, class_weights, dataset_labels
from fer_project.metrics import collect_predictions, plot_confusion_matrix, save_classification_report
from fer_project.models import build_model
from fer_project.training import fit

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE


## 2. Experiment Configurations

In [ ]:
EXPERIMENTS = [
    {
        'name': 'baseline_cnn',
        'model_kind': 'baseline_cnn',
        'train_config': TransformConfig(image_size=48, channels=1, augment=False),
        'eval_config': TransformConfig(image_size=48, channels=1, augment=False),
        'lr': 1e-3,
        'epochs': 20,
        'use_class_weights': False,
    },
    {
        'name': 'baseline_cnn_aug',
        'model_kind': 'baseline_cnn',
        'train_config': TransformConfig(image_size=48, channels=1, augment=True),
        'eval_config': TransformConfig(image_size=48, channels=1, augment=False),
        'lr': 1e-3,
        'epochs': 20,
        'use_class_weights': False,
    },
    {
        'name': 'resnet18_frozen',
        'model_kind': 'transfer',
        'transfer_model': 'resnet18',
        'freeze_backbone': True,
        'train_config': TransformConfig(image_size=224, channels=3, augment=True, imagenet_norm=True),
        'eval_config': TransformConfig(image_size=224, channels=3, augment=False, imagenet_norm=True),
        'lr': 1e-3,
        'epochs': 10,
        'use_class_weights': False,
    },
    {
        'name': 'resnet18_finetune',
        'model_kind': 'transfer',
        'transfer_model': 'resnet18',
        'freeze_backbone': False,
        'train_config': TransformConfig(image_size=224, channels=3, augment=True, imagenet_norm=True),
        'eval_config': TransformConfig(image_size=224, channels=3, augment=False, imagenet_norm=True),
        'lr': 1e-4,
        'epochs': 10,
        'use_class_weights': False,
    },
    {
        'name': 'baseline_cnn_aug_class_weights',
        'model_kind': 'baseline_cnn',
        'train_config': TransformConfig(image_size=48, channels=1, augment=True),
        'eval_config': TransformConfig(image_size=48, channels=1, augment=False),
        'lr': 1e-3,
        'epochs': 20,
        'use_class_weights': True,
    },
]

## 3. Train One Experiment

Start with a tiny smoke test to confirm the code path works: one epoch, a small class-stratified subset, and `num_workers=0`. Then increase `subset_fraction` and epochs gradually before running the full loop.


In [ ]:
def run_experiment(config, batch_size=128, num_workers=0, weighted_sampler=False, subset_fraction=1.0):
    name = config['name']
    print(f'Running {name}')

    loaders, datasets = build_imagefolder_dataloaders(
        DATA_DIR,
        train_config=config['train_config'],
        eval_config=config['eval_config'],
        batch_size=batch_size,
        num_workers=num_workers,
        weighted_sampler=weighted_sampler,
        val_fraction=0.1,
        subset_fraction=subset_fraction,
        seed=42,
    )
    print({split: len(dataset) for split, dataset in datasets.items()})

    if config['model_kind'] == 'baseline_cnn':
        model = build_model('baseline_cnn')
    else:
        model = build_model(
            'transfer',
            transfer_model=config.get('transfer_model', 'resnet18'),
            freeze_backbone=config.get('freeze_backbone', True),
        )

    weights = class_weights(dataset_labels(datasets['train'])) if config.get('use_class_weights') else None
    checkpoint_path = RESULTS_DIR / 'checkpoints' / f'{name}.pt'
    history = fit(
        model,
        loaders,
        device=DEVICE,
        epochs=config['epochs'],
        lr=config['lr'],
        class_weight=weights,
        checkpoint_path=checkpoint_path,
    )

    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    y_true, y_pred = collect_predictions(model, loaders['test'], DEVICE)

    report_path = RESULTS_DIR / 'metrics' / f'{name}_classification_report.csv'
    report = save_classification_report(y_true, y_pred, report_path)
    plot_confusion_matrix(y_true, y_pred, RESULTS_DIR / 'figures' / f'{name}_confusion_matrix.png')

    history_frame = pd.DataFrame(history)
    history_frame.to_csv(RESULTS_DIR / 'metrics' / f'{name}_history.csv', index=False)
    return history_frame, report

# First smoke test for code validation:
# smoke_config = {**EXPERIMENTS[0], 'epochs': 1}
# history, report = run_experiment(smoke_config, batch_size=32, num_workers=0, subset_fraction=0.05)
# report


## 4. Run All Experiments

In [ ]:
all_results = {}

for config in EXPERIMENTS:
    history, report = run_experiment(config)
    all_results[config['name']] = {
        'best_val_accuracy': float(history['val_accuracy'].max()),
        'test_accuracy': float(report.loc['accuracy', 'precision']),
        'test_macro_f1': float(report.loc['macro avg', 'f1-score']),
        'test_weighted_f1': float(report.loc['weighted avg', 'f1-score']),
    }

summary = pd.DataFrame(all_results).T.sort_values('test_macro_f1', ascending=False)
summary.to_csv(RESULTS_DIR / 'metrics' / 'experiment_summary.csv')
summary

## 5. Report Checklist

- Compare validation curves for convergence and overfitting.
- Rank models by macro F1 and weighted F1, not only accuracy.
- Use confusion matrices to discuss confused emotions.
- Explain the transfer learning input adaptation: grayscale 48x48 to RGB 224x224.
- Discuss whether augmentation, fine-tuning, or class weights helped.